## Multi-Layered Perceptrons
## Apna khud ka Neural Framework banana

> Yeh notebook [AI for Beginners Curricula](http://github.com/microsoft/ai-for-beginners) ka hissa hai. Poora learning material dekhne ke liye repository par jaiye.

Is notebook mein hum dheere-dheere apna khud ka neural framework banayenge, jo multi-class classification tasks aur multi-layered perceptrons ke saath regression solve kar sakega.

Sabse pehle, kuch zaroori libraries import karte hain.

In [ ]:
%matplotlib nbagg
import matplotlib.pyplot as plt 
from matplotlib import gridspec
from sklearn.datasets import make_classification
import numpy as np
# pick the seed for reproducibility - change it to explore the effects of random variations
np.random.seed(0)
import random

## Sample Dataset

Jaise pehle kiya tha, hum do parameters wale ek simple sample dataset se shuru karenge.

In [ ]:
n = 100
X, Y = make_classification(n_samples = n, n_features=2,
                           n_redundant=0, n_informative=2, flip_y=0.2)
X = X.astype(np.float32)
Y = Y.astype(np.int32)

# Split into train and test dataset
train_x, test_x = np.split(X, [n*8//10])
train_labels, test_labels = np.split(Y, [n*8//10])

In [ ]:
def plot_dataset(suptitle, features, labels):
    # prepare the plot
    fig, ax = plt.subplots(1, 1)
    #pylab.subplots_adjust(bottom=0.2, wspace=0.4)
    fig.suptitle(suptitle, fontsize = 16)
    ax.set_xlabel('$x_i[0]$ -- (feature 1)')
    ax.set_ylabel('$x_i[1]$ -- (feature 2)')

    colors = ['r' if l else 'b' for l in labels]
    ax.scatter(features[:, 0], features[:, 1], marker='o', c=colors, s=100, alpha = 0.5)
    fig.show()

In [ ]:
plot_dataset('Scatterplot of the training data', train_x, train_labels)
plt.show()

In [ ]:
print(train_x[:5])
print(train_labels[:5])

## Machine Learning ki problem

Maan lijiye humare paas input dataset $\langle X,Y\rangle$ hai, jahan $X$ features ka set hai aur $Y$ - unke corresponding labels. Regression problem ke liye, $y_i\in\mathbb{R}$ hota hai, aur classification ke liye ise ek class number $y_i\in\{0,\dots,n\}$ se dikhaya jaata hai.

Koi bhi machine learning model ek function $f_\theta(x)$ se represent kiya ja sakta hai, jahan $\theta$ **parameters** ka set hai. Hamara goal aise parameters $\theta$ dhoondhna hai jisse hamara model dataset ko sabse acche tareeke se fit kar sake. Iska criterion **loss function** $\mathcal{L}$ se define hota hai, aur humein uski optimal value dhoondhni hai:

$$
\theta = \mathrm{argmin}_\theta \mathcal{L}(f_\theta(X),Y)
$$

Loss function is par depend karta hai ki kaunsi problem solve ho rahi hai.

### Regression ke liye Loss Functions

Regression ke liye, hum aksar **absolute error** $\mathcal{L}_{abs}(\theta) = \sum_{i=1}^n |y_i - f_{\theta}(x_i)|$, ya **mean squared error**: $\mathcal{L}_{sq}(\theta) = \sum_{i=1}^n (y_i - f_{\theta}(x_i))^2$ use karte hain.

In [ ]:
# helper function for plotting various loss functions
def plot_loss_functions(suptitle, functions, ylabels, xlabel):
    fig, ax = plt.subplots(1,len(functions), figsize=(9, 3))
    plt.subplots_adjust(bottom=0.2, wspace=0.4)
    fig.suptitle(suptitle)
    for i, fun in enumerate(functions):
        ax[i].set_xlabel(xlabel)
        if len(ylabels) > i:
            ax[i].set_ylabel(ylabels[i])
        ax[i].plot(x, fun)
    plt.show()

In [ ]:
x = np.linspace(-2, 2, 101)
plot_loss_functions(
    suptitle = 'Common loss functions for regression',
    functions = [np.abs(x), np.power(x, 2)],
    ylabels   = ['$\mathcal{L}_{abs}}$ (absolute loss)',
                 '$\mathcal{L}_{sq}$ (squared loss)'],
    xlabel    = '$y - f(x_i)$')

### Classification ke liye Loss Functions

Chaliye kuch der ke liye binary classification ke baare mein sochte hain. Is case mein humare paas do classes hote hain, jinhe 0 aur 1 number diya gaya hai. Network ka output $f_\theta(x_i)\in [0,1]$ basically class 1 choose karne ki probability define karta hai.

**0-1 Loss**

0-1 loss model ki accuracy calculate karne jaisa hi hai - hum sahi classifications ki ginti karte hain:

$$\mathcal{L}_{0-1} = \sum_{i=1}^n l_i \quad  l_i = \begin{cases}
         0 & (f(x_i)<0.5 \land y_i=0) \lor (f(x_i)<0.5 \land y_i=1) \\
         1 & \mathrm{ otherwise}
       \end{cases} \\
$$

Lekin accuracy khud yeh nahi batati ki hum sahi classification se kitne door hain. Ho sakta hai humne sahi class thodi si hi miss ki ho, aur yeh ek tarah se "better" hai (matlab humein weights ko bahut kam sudharna padega) us case ke muqable jab hum kaafi zyada miss karte hain. Isliye, aksar logistic loss use hota hai, jo yeh baat dhyan mein rakhta hai.

**Logistic Loss**

$$\mathcal{L}_{log} = \sum_{i=1}^n -y\log(f_{\theta}(x_i)) - (1-y)\log(1-f_\theta(x_i))$$

In [ ]:
x = np.linspace(0,1,100)
def zero_one(d):
    if d < 0.5:
        return 0
    return 1
zero_one_v = np.vectorize(zero_one)

def logistic_loss(fx):
    # assumes y == 1
    return -np.log(fx)

In [ ]:
plot_loss_functions(suptitle = 'Common loss functions for classification (class=1)',
                   functions = [zero_one_v(x), logistic_loss(x)],
                   ylabels    = ['$\mathcal{L}_{0-1}}$ (0-1 loss)',
                                 '$\mathcal{L}_{log}$ (logistic loss)'],
                   xlabel     = '$p$')


Logistic loss ko samajhne ke liye, expected output ke do cases dekhte hain:
* Agar hum output 1 ($y=1$) expect karte hain, toh loss $-log f_\theta(x_i)$ hoga. Loss 0 tab hoga jab network 1 ko 1 ki probability ke saath predict kare, aur jab 1 ki probability kam hoti hai toh loss badhta hai.
* Agar hum output 0 ($y=0$) expect karte hain, toh loss $-log(1-f_\theta(x_i))$ hoga. Yahan, $1-f_\theta(x_i)$ woh probability hai jo network ne 0 ke liye predict ki hai, aur log-loss ka matlab wahi hai jo pichle case mein bataya gaya tha.

## Neural Network Architecture

Humne binary classification problem ke liye ek dataset taiyaar kiya hai. Lekin shuru se hi ise multi-class classification ki tarah treat karte hain, taaki hum apne code ko aasaani se multi-class classification mein badal sakein. Is case mein, hamare one-layer perceptron ki architecture kuch aisi hogi:

Network ke do outputs do classes ke corresponding hote hain, aur do outputs mein se jiski value sabse zyada hoti hai, woh class sahi solution ke roop mein choose ki jaati hai.

Model ko is tarah define kiya gaya hai:
$$
f_\theta(x) = W\times x + b
$$
jahan $$\theta = \langle W,b\rangle$$ parameters hain.

Hum is linear layer ko ek Python class ke roop mein define karenge, jisme ek `forward` function hoga jo calculation karta hai. Yeh input value $x$ leta hai aur layer ka output produce karta hai. Parameters `W` aur `b` layer class ke andar store hote hain, aur construction ke time par unhe respectively random values aur zero se initialize kiya jaata hai.

In [ ]:
class Linear:
    def __init__(self,nin,nout):
        self.W = np.random.normal(0, 1.0/np.sqrt(nin), (nout, nin))
        self.b = np.zeros((1,nout))
        
    def forward(self, x):
        return np.dot(x, self.W.T) + self.b
    
net = Linear(2,2)
net.forward(train_x[0:5])

Kai cases mein, ek input value par kaam karne ki jagah, input values ke vector par kaam karna zyada efficient hota hai. Kyunki hum Numpy operations use karte hain, hum apne network ko input values ka vector pass kar sakte hain, aur yeh humein output values ka vector dega.

## Softmax: Output ko probabilities mein badalna

Jaisa ki aap dekh sakte hain, hamare outputs probabilities nahi hain - yeh koi bhi value le sakte hain. Inhe probabilities mein badalne ke liye, humein saari classes ke beech values ko normalize karna hoga. Yeh **softmax** function use karke kiya jaata hai:
$$\sigma(\mathbf{z}_c) = \frac{e^{z_c}}{\sum_{j} e^{z_j}}, \quad\mathrm{for}\quad c\in 1 .. |C|$$

<img src="https://raw.githubusercontent.com/shwars/NeuroWorkshop/master/images/NeuroArch-softmax.PNG" width="50%">

> Network ke output $\sigma(\mathbf{z})$ ko classes $C$ ke set par probability distribution ki tarah interpret kiya ja sakta hai: $q = \sigma(\mathbf{z}_c) = \hat{p}(c | x)$

Hum `Softmax` layer ko usi tarah define karenge, ek class ki tarah jisme `forward` function hoga:

In [ ]:
class Softmax:
    def forward(self,z):
        zmax = z.max(axis=1,keepdims=True)
        expz = np.exp(z-zmax)
        Z = expz.sum(axis=1,keepdims=True)
        return expz / Z

softmax = Softmax()
softmax.forward(net.forward(train_x[0:10]))

Aap dekh sakte hain ki ab humein outputs ke roop mein probabilities mil rahi hain, yaani har output vector ka sum bilkul 1 hai.

Agar humare paas 2 se zyada classes hon, toh softmax saari classes ke beech probabilities ko normalize kar dega. Yeh ek network architecture ka diagram hai, jo MNIST digit classification karta hai:

![MNIST Classifier](../../../../../lessons/3-NeuralNetworks/04-OwnFramework/images/Cross-Entropy-Loss.PNG)

## Cross-Entropy Loss

Classification mein loss function aam taur par ek logistic function hota hai, jise **cross-entropy loss** ke roop mein generalize kiya ja sakta hai. Cross-entropy loss ek aisa function hai jo do arbitrary probability distributions ke beech similarity calculate kar sakta hai. Iske baare mein aur detail mein aap [Wikipedia par](https://en.wikipedia.org/wiki/Cross_entropy) dekh sakte hain.

Hamare case mein, pehli distribution hamare network ka probability output hai, aur doosri **one-hot** distribution hai, jo batati hai ki given class $c$ ki probability 1 hai (baaki sab 0 hain). Is situation mein, cross-entropy loss ko $-\log p_c$ ke roop mein calculate kiya ja sakta hai, jahan $c$ expected class hai, aur $p_c$ woh probability hai jo hamare neural network ne is class ke liye di hai.

> Agar network expected class ke liye probability 1 return karta hai, toh cross-entropy loss 0 hoga. Actual class ki probability jitni 0 ke kareeb hogi, cross-entropy loss utna hi zyada hoga (aur yeh infinity tak ja sakta hai!).

In [ ]:
def plot_cross_ent():
    p = np.linspace(0.01, 0.99, 101) # estimated probability p(y|x)
    cross_ent_v = np.vectorize(cross_ent)
    f3, ax = plt.subplots(1,1, figsize=(8, 3))
    l1, = plt.plot(p, cross_ent_v(p, 1), 'r--')
    l2, = plt.plot(p, cross_ent_v(p, 0), 'r-')
    plt.legend([l1, l2], ['$y = 1$', '$y = 0$'], loc = 'upper center', ncol = 2)
    plt.xlabel('$\hat{p}(y|x)$', size=18)
    plt.ylabel('$\mathcal{L}_{CE}$', size=18)
    plt.show()

In [ ]:
def cross_ent(prediction, ground_truth):
    t = 1 if ground_truth > 0.5 else 0
    return -t * np.log(prediction) - (1 - t) * np.log(1 - prediction)
plot_cross_ent()

Cross-entropy loss ko phir se ek alag layer ke roop mein define kiya jayega, lekin `forward` function mein do input values honge: network ki pichli layers ka output `p`, aur expected class `y`:

In [ ]:
class CrossEntropyLoss:
    def forward(self,p,y):
        self.p = p
        self.y = y
        p_of_y = p[np.arange(len(y)), y]
        log_prob = np.log(p_of_y)
        return -log_prob.mean() # average over all input samples

cross_ent_loss = CrossEntropyLoss()
p = softmax.forward(net.forward(train_x[0:10]))
cross_ent_loss.forward(p,train_labels[0:10])

Ab tak humne network ki alag-alag layers ke liye alag-alag classes define ki hain. In layers ke structure ko **computational graph** ke roop mein dikhaya ja sakta hai. Ab hum diye gaye training dataset (ya uske kisi hisse) ke liye loss ko neeche diye tareeke se calculate kar sakte hain:

In [ ]:
z = net.forward(train_x[0:10])
p = softmax.forward(z)
loss = cross_ent_loss.forward(p,train_labels[0:10])
print(loss)

## Loss Minimization problem aur Network Training

Jab humne apna network $f_\theta$ ke roop mein define kar liya, aur loss function $\mathcal{L}(Y,f_\theta(X))$ diya gaya hai, toh hum $\mathcal{L}$ ko ek fixed training dataset ke saath $\theta$ ka function maan sakte hain: $\mathcal{L}(\theta) = \mathcal{L}(Y,f_\theta(X))$

Is situation mein, network training $\mathcal{L}$ ko $\theta$ ke argument ke hisaab se minimize karne ki problem hogi:
$$
\theta = \mathrm{argmin}_{\theta} \mathcal{L}(Y,f_\theta(X))
$$

Function optimization ka ek famous tareeka **gradient descent** kehlata hai. Idea yeh hai ki hum loss function ka derivative (multi-dimensional case mein ise **gradient** kehte hain) parameters ke respect mein nikaal sakte hain, aur parameters ko is tarah badal sakte hain ki error kam ho jaye.

Gradient descent is tarah kaam karta hai:
 * Parameters ko kuch random values $w^{(0)}$, $b^{(0)}$ se initialize karein.
 * Neeche diya gaya step kai baar repeat karein:

 $$\begin{align}
 W^{(i+1)}&=W^{(i)}-\eta\frac{\partial\mathcal{L}}{\partial W}\\
 b^{(i+1)}&=b^{(i)}-\eta\frac{\partial\mathcal{L}}{\partial b}
 \end{align}
 $$

Training ke dauran, optimization steps poore dataset ko dhyan mein rakhkar kiye jaane chahiye (yaad rakhiye ki loss saare training samples par sum/average ke roop mein calculate hota hai). Lekin real life mein hum dataset ke chhote hisse lete hain jinhe **minibatches** kehte hain, aur data ke ek subset ke basis par gradients calculate karte hain. Kyunki har baar subset random tareeke se liya jaata hai, is method ko **stochastic gradient descent** (SGD) kehte hain.

## Backward Propagation

<img src="images/ComputeGraph.png" width="300px" align="left"/>

$$\def\L{\mathcal{L}}\def\zz#1#2{\frac{\partial#1}{\partial#2}}
\begin{align}
\zz{\L}{W} =& \zz{\L}{p}\zz{p}{z}\zz{z}{W}\cr
\zz{\L}{b} =& \zz{\L}{p}\zz{p}{z}\zz{z}{b}
\end{align}
$$

Yeh process network ke output se uske parameters tak loss error ko wapas distribute karne se shuru hota hai. Isliye is process ko **backpropagation** kehte hain.

Network training ka ek pass do hisson mein hota hai:
* **Forward pass**, jisme hum diye gaye input minibatch ke liye loss function ki value calculate karte hain
* **Backward pass**, jisme hum is error ko kam karne ki koshish karte hain aur ise computational graph ke through model parameters tak wapas distribute karte hain.

### Backpropagation ka Implementation

* Chaliye har node mein `backward` function add karte hain, jo derivatives calculate karega aur backward pass ke dauran error ko aage badhayega.
* Humein upar bataye gaye process ke mutabik parameter updates bhi implement karne honge.

Humein har layer ke liye derivatives manually calculate karne honge, jaise ki linear layer $z = x\times W+b$ ke liye:
$$\begin{align}
\frac{\partial z}{\partial W} &= x \\
\frac{\partial z}{\partial b} &= 1 \\
\end{align}$$

Agar humein layer ke output par error $\Delta z$ ko compensate karna hai, toh humein weights ko is tarah update karna hoga:
$$\begin{align}
\Delta x &= \Delta z \times W \\
\Delta W &= \frac{\partial z}{\partial W} \Delta z = \Delta z \times x \\
\Delta b &= \frac{\partial z}{\partial b} \Delta z = \Delta z \\
\end{align}$$

**Important:** Calculations har training sample ke liye alag-alag nahi ki jaati, balki poore **minibatch** ke liye ki jaati hain. Zaroori parameter updates $\Delta W$ aur $\Delta b$ poore minibatch ke liye calculate hote hain, aur corresponding vectors ke dimensions hote hain: $x\in\mathbb{R}^{\mathrm{minibatch}\, \times\, \mathrm{nclass}}$

In [ ]:
class Linear:
    def __init__(self,nin,nout):
        self.W = np.random.normal(0, 1.0/np.sqrt(nin), (nout, nin))
        self.b = np.zeros((1,nout))
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        
    def forward(self, x):
        self.x=x
        return np.dot(x, self.W.T) + self.b
    
    def backward(self, dz):
        dx = np.dot(dz, self.W)
        dW = np.dot(dz.T, self.x)
        db = dz.sum(axis=0)
        self.dW = dW
        self.db = db
        return dx
    
    def update(self,lr):
        self.W -= lr*self.dW
        self.b -= lr*self.db

Isi tarah hum apni baaki layers ke liye bhi `backward` function define kar sakte hain:

In [ ]:
class Softmax:
    def forward(self,z):
        self.z = z
        zmax = z.max(axis=1,keepdims=True)
        expz = np.exp(z-zmax)
        Z = expz.sum(axis=1,keepdims=True)
        return expz / Z
    def backward(self,dp):
        p = self.forward(self.z)
        pdp = p * dp
        return pdp - p * pdp.sum(axis=1, keepdims=True)
    
class CrossEntropyLoss:
    def forward(self,p,y):
        self.p = p
        self.y = y
        p_of_y = p[np.arange(len(y)), y]
        log_prob = np.log(p_of_y)
        return -log_prob.mean()
    def backward(self,loss):
        dlog_softmax = np.zeros_like(self.p)
        dlog_softmax[np.arange(len(self.y)), self.y] -= 1.0/len(self.y)
        return dlog_softmax / self.p

## Model ko Train karna

Ab hum **training loop** likhne ke liye ready hain, jo hamare dataset mein se guzrega aur minibatch dar minibatch optimization karega. Dataset mein se ek poora pass aksar **epoch** kehlata hai:

In [ ]:
lin = Linear(2,2)
softmax = Softmax()
cross_ent_loss = CrossEntropyLoss()

learning_rate = 0.1

pred = np.argmax(lin.forward(train_x),axis=1)
acc = (pred==train_labels).mean()
print("Initial accuracy: ",acc)

batch_size=4
for i in range(0,len(train_x),batch_size):
    xb = train_x[i:i+batch_size]
    yb = train_labels[i:i+batch_size]
    
    # forward pass
    z = lin.forward(xb)
    p = softmax.forward(z)
    loss = cross_ent_loss.forward(p,yb)
    
    # backward pass
    dp = cross_ent_loss.backward(loss)
    dz = softmax.backward(dp)
    dx = lin.backward(dz)
    lin.update(learning_rate)
    
pred = np.argmax(lin.forward(train_x),axis=1)
acc = (pred==train_labels).mean()
print("Final accuracy: ",acc)
    

Model ki accuracy ko lagbhag 50% se badhkar sirf ek epoch mein 80% tak jaate dekhna accha lagta hai.

## Network Class

Kyunki kai cases mein neural network sirf layers ka combination hota hai, hum ek class bana sakte hain jo humein layers ko saath jodne aur unke through forward aur backward pass karne dega, bina us logic ko explicitly program kiye. Hum `Net` class ke andar layers ki list store karenge aur nayi layers add karne ke liye `add()` function use karenge:

In [ ]:
class Net:
    def __init__(self):
        self.layers = []
    
    def add(self,l):
        self.layers.append(l)
        
    def forward(self,x):
        for l in self.layers:
            x = l.forward(x)
        return x
    
    def backward(self,z):
        for l in self.layers[::-1]:
            z = l.backward(z)
        return z
    
    def update(self,lr):
        for l in self.layers:
            if 'update' in l.__dir__():
                l.update(lr)

Is `Net` class ke saath hamara model definition aur training zyada streamlined ho jaata hai:

In [ ]:
net = Net()
net.add(Linear(2,2))
net.add(Softmax())
loss = CrossEntropyLoss()

def get_loss_acc(x,y,loss=CrossEntropyLoss()):
    p = net.forward(x)
    l = loss.forward(p,y)
    pred = np.argmax(p,axis=1)
    acc = (pred==y).mean()
    return l,acc

print("Initial loss={}, accuracy={}: ".format(*get_loss_acc(train_x,train_labels)))

def train_epoch(net, train_x, train_labels, loss=CrossEntropyLoss(), batch_size=4, lr=0.1):
    for i in range(0,len(train_x),batch_size):
        xb = train_x[i:i+batch_size]
        yb = train_labels[i:i+batch_size]

        p = net.forward(xb)
        l = loss.forward(p,yb)
        dp = loss.backward(l)
        dx = net.backward(dp)
        net.update(lr)
 
train_epoch(net,train_x,train_labels)
        
print("Final loss={}, accuracy={}: ".format(*get_loss_acc(train_x,train_labels)))
print("Test loss={}, accuracy={}: ".format(*get_loss_acc(test_x,test_labels)))

## Training process ka Plot banana

Yeh dekhna accha hoga ki network kaise train ho raha hai! Iske liye hum ek `train_and_plot` function define karenge. Network ki state ko visualize karne ke liye hum level map use karenge, yaani network output ki alag-alag values ko alag-alag colors ke through dikhayenge.

> Agar neeche diye gaye plotting code ka kuch hissa samajh na aaye toh chinta mat kijiye - underlying neural network concepts samajhna zyada important hai.

In [ ]:
def train_and_plot(n_epoch, net, loss=CrossEntropyLoss(), batch_size=4, lr=0.1):
    fig, ax = plt.subplots(2, 1)
    ax[0].set_xlim(0, n_epoch + 1)
    ax[0].set_ylim(0,1)

    train_acc = np.empty((n_epoch, 3))
    train_acc[:] = np.NAN
    valid_acc = np.empty((n_epoch, 3))
    valid_acc[:] = np.NAN

    for epoch in range(1, n_epoch + 1):

        train_epoch(net,train_x,train_labels,loss,batch_size,lr)
        tloss, taccuracy = get_loss_acc(train_x,train_labels,loss)
        train_acc[epoch-1, :] = [epoch, tloss, taccuracy]
        vloss, vaccuracy = get_loss_acc(test_x,test_labels,loss)
        valid_acc[epoch-1, :] = [epoch, vloss, vaccuracy]
        
        ax[0].set_ylim(0, max(max(train_acc[:, 2]), max(valid_acc[:, 2])) * 1.1)

        plot_training_progress(train_acc[:, 0], (train_acc[:, 2],
                                                 valid_acc[:, 2]), fig, ax[0])
        plot_decision_boundary(net, fig, ax[1])
        fig.canvas.draw()
        fig.canvas.flush_events()

    return train_acc, valid_acc

In [ ]:
import matplotlib.cm as cm

def plot_decision_boundary(net, fig, ax):
    draw_colorbar = True
    # remove previous plot
    while ax.collections:
        ax.collections.pop()
        draw_colorbar = False

    # generate countour grid
    x_min, x_max = train_x[:, 0].min() - 1, train_x[:, 0].max() + 1
    y_min, y_max = train_x[:, 1].min() - 1, train_x[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    grid_points = np.c_[xx.ravel().astype('float32'), yy.ravel().astype('float32')]
    n_classes = max(train_labels)+1
    while train_x.shape[1] > grid_points.shape[1]:
        # pad dimensions (plot only the first two)
        grid_points = np.c_[grid_points,
                            np.empty(len(xx.ravel())).astype('float32')]
        grid_points[:, -1].fill(train_x[:, grid_points.shape[1]-1].mean())

    # evaluate predictions
    prediction = np.array(net.forward(grid_points))
    # for two classes: prediction difference
    if (n_classes == 2):
        Z = np.array([0.5+(p[0]-p[1])/2.0 for p in prediction]).reshape(xx.shape)
    else:
        Z = np.array([p.argsort()[-1]/float(n_classes-1) for p in prediction]).reshape(xx.shape)
    
    # draw contour
    levels = np.linspace(0, 1, 40)
    cs = ax.contourf(xx, yy, Z, alpha=0.4, levels = levels)
    if draw_colorbar:
        fig.colorbar(cs, ax=ax, ticks = [0, 0.5, 1])
    c_map = [cm.jet(x) for x in np.linspace(0.0, 1.0, n_classes) ]
    colors = [c_map[l] for l in train_labels]
    ax.scatter(train_x[:, 0], train_x[:, 1], marker='o', c=colors, s=60, alpha = 0.5)

In [ ]:
def plot_training_progress(x, y_data, fig, ax):
    styles = ['k--', 'g-']
    # remove previous plot
    while ax.lines:
        ax.lines.pop()
    # draw updated lines
    for i in range(len(y_data)):
        ax.plot(x, y_data[i], styles[i])
    ax.legend(ax.lines, ['training accuracy', 'validation accuracy'],
              loc='upper center', ncol = 2)

In [ ]:
%matplotlib nbagg 
net = Net()
net.add(Linear(2,2))
net.add(Softmax())

res = train_and_plot(30,net,lr=0.005)

Upar wala cell run karne ke baad, aap interactively dekh payenge ki training ke dauran classes ke beech ki boundary kaise badalti hai. Dhyan rakhiye ki humne bahut chhota learning rate choose kiya hai taaki hum dekh sakein ki yeh process kaise hota hai.

## Multi-Layer Models

Upar wala network kai layers se banaya gaya hai, lekin usme sirf ek `Linear` layer thi, jo actual classification karti hai. Agar hum aisi kai layers add karne ka decide karein toh kya hoga?

Hairat ki baat hai, hamara code kaam karega! Lekin ek bahut important baat dhyan dene wali hai ki linear layers ke beech humein ek non-linear **activation function** jaise `tanh` use karna hoga. Is non-linearity ke bina, kai linear layers ki expressive power sirf ek layer jitni hi hogi - kyunki linear functions ka combination bhi linear hi hota hai!

In [ ]:
class Tanh:
    def forward(self,x):
        y = np.tanh(x)
        self.y = y
        return y
    def backward(self,dy):
        return (1.0-self.y**2)*dy

Kai layers add karna samajhdaari hai, kyunki one-layer network ke ulat, multi-layer model un sets ko accurately classify kar sakta hai jo linearly separable nahi hain. Yaani, kai layers wala model **zyada capable** hoga.

> Yeh dikhaya ja sakta hai ki kaafi saare neurons ke saath, two-layer model kisi bhi convex dataset ko classify kar sakta hai, aur three-layer network lagbhag kisi bhi set ko classify kar sakta hai.

Mathematically, multi-layer perceptron ko ek zyada complex function $f_\theta$ se represent kiya jayega, jise kai steps mein calculate kiya ja sakta hai:
* $z_1 = W_1\times x+b_1$
* $z_2 = W_2\times\alpha(z_1)+b_2$
* $f = \sigma(z_2)$

Yahan, $\alpha$ ek **non-linear activation function** hai, $\sigma$ ek softmax function hai, aur $\theta=\langle W_1,b_1,W_2,b_2\rangle$ parameters hain.

Gradient descent algorithm wahi rahega, lekin gradients calculate karna zyada mushkil hoga. Chain differentiation rule ko dhyan mein rakhte hue, hum derivatives ko is tarah calculate kar sakte hain:

$$\begin{align}
\frac{\partial\mathcal{L}}{\partial W_2} &= \color{red}{\frac{\partial\mathcal{L}}{\partial\sigma}\frac{\partial\sigma}{\partial z_2}}\color{black}{\frac{\partial z_2}{\partial W_2}} \\
\frac{\partial\mathcal{L}}{\partial W_1} &= \color{red}{\frac{\partial\mathcal{L}}{\partial\sigma}\frac{\partial\sigma}{\partial z_2}}\color{black}{\frac{\partial z_2}{\partial\alpha}\frac{\partial\alpha}{\partial z_1}\frac{\partial z_1}{\partial W_1}}
\end{align}
$$

Dhyan dijiye ki in sabhi expressions ki shuruaat same rehti hai, aur isliye hum ek linear layer se aage backpropagation continue kar sakte hain taaki computational graph mein aage ke weights ko adjust kiya ja sake.

Ab chaliye two-layer network ke saath experiment karte hain:

In [ ]:
net = Net()
net.add(Linear(2,10))
net.add(Tanh())
net.add(Linear(10,2))
net.add(Softmax())
loss = CrossEntropyLoss()

In [ ]:
res = train_and_plot(30,net,lr=0.01)

## Hamesha Multi-Layered model use kyun nahi karte?

Humne dekha hai ki multi-layered model one-layered model ke muqable zyada *powerful* aur *expressive* hota hai. Aap soch sakte hain ki hum hamesha multi-layered model kyun use nahi karte. Is sawaal ka jawab hai **overfitting**.

Hum is term par aage ke sections mein aur baat karenge, lekin iska main idea yeh hai: **Model jitna zyada powerful hoga, woh training data ko utni hi acchi tarah approximate kar sakta hai, aur naye data par, jo usne pehle nahi dekha, sahi tarah generalize karne ke liye use utna hi zyada data chahiye hoga.**

**Ek linear model:**
* Humein high training loss milne ki sambhavna hai - ise **underfitting** kehte hain, jab model ke paas saara data sahi tarah alag karne ki kaafi capacity nahi hoti.
* Validation loss aur training loss lagbhag same hote hain. Model se test data par acchi performance ki ummeed ki ja sakti hai.

**Complex multi-layered model:**
* Low training loss - model training data ko acchi tarah approximate kar sakta hai, kyunki uske paas kaafi expressive capacity hoti hai.
* Validation loss training loss se kahin zyada ho sakta hai aur training ke dauran badhne bhi lag sakta hai - aisa isliye hota hai kyunki model "training points" ko yaad kar leta hai aur "bigger picture" kho deta hai.

![Overfitting](../../../../../lessons/3-NeuralNetworks/04-OwnFramework/images/overfit.png)

> Is picture mein, `x` training data ko dikhata hai, `o` - validation data. Left mein - linear model (one-layered), yeh data ki nature ko kaafi acchi tarah approximate karta hai. Right mein - overfitted model, yeh model training data ko perfectly approximate karta hai, lekin kisi aur data ke saath iska koi matlab nahi banta (validation error bahut zyada hai).

## Key Takeaways

* Simple models (kam layers, kam neurons) jinme parameters ki sankhya kam hoti hai ("low capacity"), unke overfit hone ki sambhavna kam hoti hai.
* Zyada complex models (zyada layers, har layer mein zyada neurons, high capacity) ke overfit hone ki sambhavna zyada hoti hai. Humein validation error monitor karna chahiye taaki yeh zyada training ke saath badhna shuru na kar de.
* Zyada complex models ko training ke liye zyada data chahiye hota hai.
* Aap overfitting ki problem ko neeche diye tareekon se solve kar sakte hain:
    - Apne model ko simple banakar
    - Training data ki matra badhakar
* **Bias-variance trade-off** ek aisa term hai jo dikhata hai ki aapko balance banana padega:
    - Model ki power aur data ki matra ke beech,
    - Overfitting aur underfitting ke beech
* Yeh decide karne ka koi ek tareeka nahi hai ki aapko kitni layers ya parameters chahiye - sabse accha tareeka hai experiment karna.

## Credits

Yeh notebook [AI for Beginners Curricula](http://github.com/microsoft/ai-for-beginners) ka hissa hai, aur ise [Dmitry Soshnikov](http://soshnikov.com) ne taiyaar kiya hai. Yeh Microsoft Research Cambridge ke Neural Network Workshop se inspired hai. Kuch code aur illustrative material [Katja Hoffmann](https://www.microsoft.com/en-us/research/people/kahofman/), [Matthew Johnson](https://www.microsoft.com/en-us/research/people/matjoh/) aur [Ryoto Tomioka](https://www.microsoft.com/en-us/research/people/ryoto/) ki presentations se liya gaya hai, aur [NeuroWorkshop](http://github.com/shwars/NeuroWorkshop) repository se bhi.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->